In [1]:
import os
import pandas as pd
import chromadb
import requests

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
import os
from pypdf import PdfReader

BASE_DIR = os.path.dirname(os.getcwd())

PDF_PATH = os.path.join(BASE_DIR, "dataset", "sample.pdf")

reader = PdfReader(PDF_PATH)

extracted_text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        extracted_text += page_text + "\n"

print("PDF extraction completed!")
print("Total pages:", len(reader.pages))
print("Total characters:", len(extracted_text))

PDF extraction completed!
Total pages: 187
Total characters: 298400


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
import os

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(extracted_text)

chunks_df = pd.DataFrame({
    "chunk_id": range(len(chunks)),
    "text": chunks
})

chunks_df.to_csv(
    os.path.join(BASE_DIR, "dataset", "chunks.csv"),
    index=False
)

print("Text splitting completed!")
print("Total chunks:", len(chunks))
print("First chunk:\n", chunks[0])

Text splitting completed!
Total chunks: 709
First chunk:
 1
Introduction
Introduction to Artificial Intelligence
By Nikhil Sharma, Josh Hug, Jacky Liang, and Henry Zhu.
Additional Contributions by Regina Wang, Peyrin Kao, Matei Gardea, Danial Toktarbayev, Catherine Chu,
Samantha Huang, Saathvik Selvan, Pranav Muralikrishnan, and Wesley Zheng.
Website Established By Wesley Zheng.
Special Thanks to Ashley Chiu.
This is the textbook for CS 188: Introduction to Artificial Intelligence at UC Berkeley. It provides an


In [4]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings generated successfully!")
print("Total embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Embeddings generated successfully!
Total embeddings: 709
Embedding dimensions: (709, 384)


In [5]:
import chromadb
import os

CHROMA_PATH = os.path.join(BASE_DIR, "chroma_db")

client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection("pdf_documents")
except Exception:
    pass

collection = client.create_collection("pdf_documents")

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings.tolist()
)

print("Data stored successfully in ChromaDB!")
print("Total stored chunks:", collection.count())

Data stored successfully in ChromaDB!
Total stored chunks: 709


In [6]:
def retrieve_chunks(question):
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )

    return results["documents"][0]


question = "What is Artificial Intelligence?"

retrieved_chunks = retrieve_chunks(question)

print("Question:", question)
print("\nTop 5 Retrieved Chunks:\n")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\nChunk {i}:")
    print(chunk[:300])
    print("-" * 50)

Question: What is Artificial Intelligence?

Top 5 Retrieved Chunks:


Chunk 1:
which maintain a model of the world and use this model to simulate performing various actions. Then, the
agent can determine hypothesized consequences of the actions and select the best one. This is simulated
“intelligence” in the sense that it’s exactly what humans do when trying to determine the b
--------------------------------------------------

Chunk 2:
introduction to the full range of topics studied in artificial intelligence, with emphasis on the “core competences”
of intelligent systems - problem solving, reasoning, decision making, and learning - and on the logical and
probabilistic foundations of these activities.
PDF Version
These notes are 
--------------------------------------------------

Chunk 3:
Last updated: October 2024
3
1.1 Agents
1.1 Agents
In artificial intelligence, the central problem at hand is the creation of a rationalagent, an entity that has
goals or preferences and tries to p

In [7]:
import requests

OLLAMA_MODEL = "llama3.2"

def generate_answer(question, context):
    prompt = f"""
Answer the question using ONLY the information in the context.

If the answer is not available, say:
"The answer is not available in the PDF."

Context:
{context}

Question:
{question}

Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=300
    )

    response.raise_for_status()
    return response.json()["response"]


context = "\n\n".join(retrieved_chunks)

answer = generate_answer(question, context)

print("Question:", question)
print("\nAnswer from PDF:")
print(answer)

Question: What is Artificial Intelligence?

Answer from PDF:
The answer is not available in the PDF.


In [8]:
def answer_without_retrieval(question):
    prompt = f"""
Answer the following question using your general knowledge.

Question: {question}

Answer:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=300
    )

    response.raise_for_status()
    return response.json()["response"]


normal_answer = answer_without_retrieval(question)

print("QUESTION:", question)

print("\nANSWER WITHOUT RETRIEVAL:")
print(normal_answer)

print("\nANSWER WITH RETRIEVAL:")
print(answer)

QUESTION: What is Artificial Intelligence?

ANSWER WITHOUT RETRIEVAL:
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning, problem-solving, decision-making, and perception.

AI technology is based on the idea that intelligence can be created and replicated in machines, allowing them to perform tasks that typically require human intelligence, such as:

1. Learning: AI systems can learn from data, improve their performance, and adapt to new situations.
2. Problem-solving: AI systems can analyze data, identify patterns, and make decisions based on that analysis.
3. Perception: AI systems can interpret and understand data from sensors, such as images, speech, and text.
4. Reasoning: AI systems can draw conclusions, make inferences, and solve problems using logic and rules.

There 

In [9]:
import pandas as pd
import os

results = pd.DataFrame([{
    "Question": question,
    "Retrieved_Chunks": "\n\n".join(retrieved_chunks),
    "Answer_With_Retrieval": answer,
    "Answer_Without_Retrieval": normal_answer
}])

answers_path = os.path.join(BASE_DIR, "dataset", "notebook_answers.csv")

results.to_csv(
    answers_path,
    index=False
)

print("Results saved successfully!")
print("Saved location:", answers_path)

results

Results saved successfully!
Saved location: c:\Users\Akshaya Nivasini\Desktop\PDF_Question_Answering\dataset\notebook_answers.csv


,Question,Retrieved_Chunks,Answer_With_Retrieval,Answer_Without_Retrieval
0,What is Artificial Intelligence?,which maintain a model of the world and use th...,The answer is not available in the PDF.,Artificial Intelligence (AI) refers to the sim...


In [10]:
questions_df = pd.read_csv(
    os.path.join(BASE_DIR, "dataset", "queries.csv")
)

all_results = []

for index, q in enumerate(questions_df["question"].dropna(), 1):

    q = str(q).strip()

    if not q:
        continue

    print(f"\nProcessing Question {index}: {q}")

    # Retrieve top 5 chunks
    retrieved = retrieve_chunks(q)
    context = "\n\n".join(retrieved)

    # Generate both answers
    rag_answer = generate_answer(q, context)
    normal_answer = answer_without_retrieval(q)

    all_results.append({
        "Question": q,
        "Retrieved_Chunks": context,
        "Answer_With_Retrieval": rag_answer,
        "Answer_Without_Retrieval": normal_answer
    })

    print("Completed!")

# Save all results
final_results = pd.DataFrame(all_results)

final_path = os.path.join(
    BASE_DIR,
    "dataset",
    "notebook_answers.csv"
)

final_results.to_csv(final_path, index=False)

print("\nALL QUESTIONS COMPLETED!")
print("Total questions:", len(final_results))
print("Results saved to:", final_path)


Processing Question 1: What is Artificial Intelligence?
Completed!

Processing Question 2: What is Machine Learning?
Completed!

Processing Question 3: What is Deep Learning?
Completed!

Processing Question 4: What is Natural Language Processing?
Completed!

Processing Question 5: What are Large Language Models?
Completed!

Processing Question 6: What is a neural network?
Completed!

Processing Question 7: What is supervised learning?
Completed!

Processing Question 8: What is unsupervised learning?
Completed!

Processing Question 9: What is reinforcement learning?
Completed!

Processing Question 10: What is an AI chatbot?
Completed!

Processing Question 11: What is the attention mechanism?
Completed!

Processing Question 12: What is a Transformer model?
Completed!

Processing Question 13: What are the applications of AI?
Completed!

Processing Question 14: What are the advantages of Artificial Intelligence?
Completed!

Processing Question 15: What are the challenges of Artificial Int